# 15 Newton's Method

| | |
|---|---|
| **Course** | Calculus I |
| **Prerequisite** | 11 |

Suppose $f(r)=0$. Follow the tangent line at the guess $x_n$ until it meets the $x$-axis. That intersection is the next guess:

$$
x_{n+1}=x_n-\dfrac{f(x_n)}{f'(x_n)}.
$$

The step requires $f'(x_n)\neq 0$, and a first guess close enough to the root that the tangent does not throw the iteration away. Each step is the linearization from lesson 11, with the zero of the line standing in for the zero of the curve.

The method approximates the root. When the root has a closed form, compare the approximation with that form. When it does not, ask that $|f(x_n)|$ become small.


## Example 1 — Square root of $2$, with a closed form beside the approximation

### Hand solution

$$
f(x)=x^2-2, \qquad f'(x)=2x.
$$

$$
x_{n+1}=x_n-\dfrac{x_n^2-2}{2x_n}=\dfrac{x_n+\dfrac{2}{x_n}}{2}.
$$

Start at $x_0=1$:

$$
\begin{aligned}
x_1 &= \dfrac{1+2}{2}=\dfrac{3}{2}, \\
x_2 &= \dfrac{\dfrac{3}{2}+\dfrac{2}{\dfrac{3}{2}}}{2}=\dfrac{\dfrac{3}{2}+\dfrac{4}{3}}{2}=\dfrac{\dfrac{17}{6}}{2}=\dfrac{17}{12}.
\end{aligned}
$$

$$
\sqrt{2}\approx 1.41421356, \qquad \dfrac{17}{12}=1.4166\ldots.
$$

Two steps have produced three correct digits.

### Python recheck


In [1]:
import sympy as sp

# f(x) = x^2 - 2, whose root is sqrt(2).
x = sp.symbols("x")
f = x**2 - 2
fp = sp.diff(f, x)

def newton(guess, steps):
    # Keep each tangent update, starting from the seed.
    values = [guess]
    for _ in range(steps):
        values.append(sp.simplify(values[-1] - f.subs(x, values[-1])/fp.subs(x, values[-1])))
    return values

# Four steps from x0 = 1, then the first two exact fractions.
seq = newton(1, 4)
for n, value in enumerate(seq):
    print(n, value, sp.N(value, 12))
assert seq[1] == sp.Rational(3, 2)
assert seq[2] == sp.Rational(17, 12)
assert abs(seq[4] - sp.sqrt(2)) < 10**(-8)
print("Confirmed")


0 1 1.00000000000
1 3/2 1.50000000000
2 17/12 1.41666666667
3 577/408 1.41421568627
4 665857/470832 1.41421356237
Confirmed


## Example 2 — A root with no simple radical formula

### Hand solution

$$
f(x)=\cos x-x, \qquad f'(x)=-\sin x-1.
$$

$$
\begin{aligned}
x_{n+1} &= x_n-\dfrac{\cos x_n-x_n}{-\sin x_n-1} \\
&= x_n+\dfrac{\cos x_n-x_n}{\sin x_n+1}.
\end{aligned}
$$

From $x_0=0.5$, one step by hand, kept to four digits:

$$
\begin{aligned}
\cos 0.5 &\approx 0.8776, \\
\sin 0.5 &\approx 0.4794, \\
x_1 &\approx 0.5+\dfrac{0.8776-0.5}{0.4794+1}=0.5+\dfrac{0.3776}{1.4794}\approx 0.755.
\end{aligned}
$$

The true root is near $0.739085$. Later steps become sharp in Python. Existence of the root comes from lesson 05: $f(0)=1>0$ and $f(1)=\cos 1-1<0$.

### Python recheck


In [2]:
import sympy as sp

# The equation cos x = x, started at 1/2.
x = sp.symbols("x")
f = sp.cos(x) - x
fp = sp.diff(f, x)
guess = sp.Rational(1, 2)
for n in range(5):
    # One Newton update, printed to 20 digits.
    guess = sp.N(guess - f.subs(x, guess)/fp.subs(x, guess), 20)
    print(n + 1, guess, "f =", sp.N(f.subs(x, guess), 8))
# A sign change on [0, 1] guarantees a root.
assert f.subs(x, 0) > 0 > f.subs(x, 1)
assert abs(f.subs(x, guess)) < 10**(-12)
print("Confirmed")


1 0.75522241710563642167 f = -0.027103312
2 0.73914166614987924495 f = -9.4615381e-5
3 0.73908513392080680328 f = -1.1809779e-9
4 0.73908513321516064177 f = -1.8380615e-19
5 0.73908513321516064166 f = 0
Confirmed


## Example 3 — Failure when the tangent is horizontal

### Hand solution

Is $f(x)=x^3-3x+3$ the right example? A simpler polynomial is

$$
f(x)=(x-1)^3+1=x^3-3x^2+3x.
$$

The useful example is one whose first guess makes the tangent exactly horizontal.

$$
\begin{aligned}
f(x) &= x^3-x-1, \\
f'(x) &= 3x^2-1=0 \quad\text{at}\quad x=\pm\dfrac{1}{\sqrt{3}}.
\end{aligned}
$$

Place the guess there and the denominator is zero, so the method is undefined. The true root lies between $1$ and $2$, because $f(1)=-1$ and $f(2)=5$. From $x_0=1.5$ the method is sound, because

$$
f'(1.5)=3\cdot 2.25-1=5.75\neq 0.
$$

### Python recheck


In [3]:
import sympy as sp

# Derivative vanishes at the bad guess 1/sqrt(3).
x = sp.symbols("x")
f = x**3 - x - 1
fp = sp.diff(f, x)
bad = 1/sp.sqrt(3)
print("derivative at the bad guess:", sp.simplify(fp.subs(x, bad)))

# Six Newton steps from the safe seed 3/2.
guess = sp.Rational(3, 2)
for n in range(6):
    guess = sp.simplify(guess - f.subs(x, guess)/fp.subs(x, guess))
print("from 1.5:", sp.N(guess, 12), "f =", sp.N(f.subs(x, guess), 5))
assert fp.subs(x, bad) == 0
assert abs(f.subs(x, guess)) < 10**(-10)
print("Confirmed")


derivative at the bad guess: 0
from 1.5: 1.32471795724 f = 1.2681e-53
Confirmed


## Example 4 — Comparison with the exact quadratic solution

### Hand solution

The equation $x^2-5x+6=0$ has roots $2$ and $3$. Newton's method from $x_0=0$, with $f'(x)=2x-5$, begins as follows.

$$
\begin{aligned}
x_1 &= 0-\dfrac{6}{-5}=\dfrac{6}{5}, \\
x_2 &= \dfrac{6}{5}-\dfrac{\dfrac{36}{25}-\dfrac{30}{5}+6}{\dfrac{12}{5}-5} \\
&= \dfrac{6}{5}-\dfrac{\dfrac{36}{25}-\dfrac{150}{25}+\dfrac{150}{25}}{\dfrac{12}{5}-\dfrac{25}{5}}.
\end{aligned}
$$

The numerator is

$$
\left(\dfrac{6}{5}\right)^2-5\cdot\dfrac{6}{5}+6=\dfrac{36}{25}-6+6=\dfrac{36}{25}.
$$

The derivative in the denominator is

$$
\dfrac{12}{5}-5=\dfrac{12-25}{5}=-\dfrac{13}{5}.
$$

$$
\begin{aligned}
x_2 &= \dfrac{6}{5}-\dfrac{\dfrac{36}{25}}{-\dfrac{13}{5}}=\dfrac{6}{5}+\dfrac{36}{25}\cdot\dfrac{5}{13} \\
&= \dfrac{6}{5}+\dfrac{36}{65}=\dfrac{78+36}{65}=\dfrac{114}{65}.
\end{aligned}
$$

Now $\dfrac{114}{65}\approx 1.753$. The iterates move toward the root $2$. The first guess lies on the left side of the parabola, and the other root is $3$.

### Python recheck


In [4]:
import sympy as sp

# Two exact Newton steps for the quadratic, starting at 0.
x = sp.symbols("x")
f = x**2 - 5*x + 6
fp = sp.diff(f, x)
guess = 0
values = [guess]
for _ in range(2):
    guess = sp.simplify(guess - f.subs(x, guess)/fp.subs(x, guess))
    values.append(guess)
print(values)
assert values[1] == sp.Rational(6, 5)
assert values[2] == sp.Rational(114, 65)
print("Confirmed")


[0, 6/5, 114/65]
Confirmed


## A pitfall

Compute one step by hand so the formula stays in your fingers. Then let Python carry the later steps exactly. If $|f|$ stays large, change the first guess. Inspect the denominator before you blame the algorithm.

## Summary

Newton's method follows the zero of the tangent line. Beside every approximation, place either the exact value or $|f(x_n)|$. A horizontal tangent stops the method.
